# AWE actual-data layer separation

Extract the **entire ZIP**, then open this notebook from the extracted folder. No network drive or original 602 MB file is required.

Included are 49 actual frames, original zero-based indices **930–979**, excluding the missing epoch documented in metadata.json, from `awe_l1r_q20_2023326T0542_00004_v02.nc`. `projected_frames.npz` contains their calibrated radiance on the same fixed 0.02° latitude/longitude grid used in our demonstration. `original_v02_matched_frames.nc` also preserves the original 300 × 300 radiance and geolocation arrays plus the other source variables for these frames.

This experiment uses the **85 km OH-layer projection from v02**. Its fast/slow labels identify apparent motion, not proven physical classes. The notebook defaults to previously fitted velocities; an optional cell re-estimates them. Contrast error against ground truth is unavailable for real data. Reconstruction RMSE alone cannot establish clean separation.

Version-specific notes: the original v02 file uses an 85 km layer and contains radiance masks. One requested epoch is missing; the gap is retained. Frame labels in plots are v02 source indices. The same geographic grid as before is retained for comparison.


In [ ]:
# Run once if needed. netCDF4 is only needed to inspect the original-frame subset.
%pip install numpy scipy matplotlib netCDF4


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, map_coordinates, maximum_filter
from scipy.signal import fftconvolve

DATA_FOLDER = Path('.')  # Change if your notebook working directory is elsewhere.
ITERATIONS = 64         # Set to 256, 512, etc. Longer runs may take several minutes.
REESTIMATE_VELOCITIES = False
OUTPUT = DATA_FOLDER / f'actual_results_{ITERATIONS}'
OUTPUT.mkdir(exist_ok=True)
data = np.load(DATA_FOLDER / 'projected_frames.npz')
I = data['radiance'].astype(float)
lat, lon = data['latitude'], data['longitude']
frames = data['frame_index']
reference_index = int(np.flatnonzero(data['matching_v01_frame_index'] == 955)[0])
times = data['epoch'] - data['epoch'][reference_index]
ny, nx = I.shape[1:]
yy, xx = np.indices((ny, nx), dtype=float)
previous = json.loads((DATA_FOLDER / 'previous_fit_metrics.json').read_text())
va = np.array(previous['slow_velocity_pixels_per_s'])
vc = np.array(previous['fast_velocity_pixels_per_s'])
assert I.shape == (49, 309, 317)
assert np.all(np.diff(times) > 0)
print('Frames:', frames[0], 'to', frames[-1], '| duration:', times[-1]-times[0], 's')
print('Slow / fast velocities [east, north], grid pixels/s:', va, vc)


## Inspect the actual observations
Values are in kRayleigh. Gray/blank areas are outside the projected footprint. The original Epoch values are retained; only relative seconds are used, avoiding ambiguity in the source time-system labels.


In [ ]:
extent = [lon[0]-.01, lon[-1]+.01, lat[0]-.01, lat[-1]+.01]
aspect = 1/np.cos(np.deg2rad(lat.mean()))
lo, hi = np.nanpercentile(I, [2, 98])
fig, axes = plt.subplots(1, 3, figsize=(14, 5), layout='constrained')
for ax, j in zip(axes, [0, reference_index, len(I)-1]):
    im = ax.imshow(I[j], origin='lower', extent=extent, aspect=aspect,
                   cmap='gray', vmin=lo, vmax=hi)
    ax.set(title=f'Frame {frames[j]}', xlabel='Longitude (degrees E)', ylabel='Latitude (degrees N)')
fig.colorbar(im, ax=axes, label='kRayleigh', shrink=.7)
plt.show()


## Matching and alignment functions
The following cell contains the original patch-matching implementation and interpolation functions. Optional motion re-estimation uses the same heuristic thresholds: patch width 31, lags 8/16/24, NCC > 0.35, slow speed < 0.18 pixels/s, fast speed > 0.22 pixels/s, averaging radius 0.09 pixels/s. These are not calibrated physical limits.


In [ ]:
def highpass(a,sigma=5):
    ok=np.isfinite(a)
    w=gaussian_filter(ok.astype(float),sigma)
    bg=gaussian_filter(np.nan_to_num(a),sigma)/np.maximum(w,1e-9)
    return np.where(ok & (w>.98),a-bg,np.nan)

def ncc(template, search):
    """Normalized cross-correlation; reject any window containing missing data."""
    ok=np.isfinite(search)
    b=np.nan_to_num(search)
    kernel=np.ones(template.shape)
    n=template.size
    count=fftconvolve(ok.astype(float),kernel,mode='valid')
    sm=fftconvolve(b,kernel,mode='valid')
    ss=fftconvolve(b*b,kernel,mode='valid')
    t=template-template.mean()
    denom=np.sqrt(np.maximum(ss-sm*sm/n,0)*np.sum(t*t))
    c=fftconvolve(b,t[::-1,::-1],mode='valid')/np.maximum(denom,1e-12)
    c[count<n-.1]=np.nan
    return c

def motions():
    H=np.array([highpass(a) for a in I])
    rows=[]
    half=15
    for lag in (8,16,24):
        radius=int(np.ceil(lag*1.1))
        for start in (0,12,25):
            end=start+lag
            if end>=len(I):continue
            dt=times[end]-times[start]
            for cy in range(half+radius,ny-half-radius,18):
                for cx in range(half+radius,nx-half-radius,18):
                    t=H[start,cy-half:cy+half+1,cx-half:cx+half+1]
                    if not np.isfinite(t).all() or t.std()<.08:continue
                    s=H[end,cy-half-radius:cy+half+radius+1,cx-half-radius:cx+half+radius+1]
                    c=ncc(t,s)
                    if not np.isfinite(c).any():continue
                    # Keep distinct local peaks, not only the winning displacement.
                    peak=(np.nan_to_num(c,nan=-2)==maximum_filter(np.nan_to_num(c,nan=-2),size=5)) & (c>.35)
                    py,px=np.where(peak)
                    order=np.argsort(c[py,px])[::-1][:4]
                    for rank,k in enumerate(order):
                        dy,dx=float(py[k]-radius),float(px[k]-radius)
                        if abs(dy)>=radius or abs(dx)>=radius:continue
                        # Subpixel quadratic interpolation about each local maximum.
                        for axis in (0,1):
                            p=int(py[k]); q=int(px[k])
                            z=c[p-1:p+2,q] if axis==0 else c[p,q-1:q+2]
                            if len(z)==3 and np.isfinite(z).all():
                                den=z[0]-2*z[1]+z[2]
                                delta=np.clip(.5*(z[0]-z[2])/den,-.5,.5) if den<0 else 0
                                if axis==0:dy+=delta
                                else:dx+=delta
                        rows.append([cx,cy,start,end,dx/dt,dy/dt,float(c[py[k],px[k]]),rank])
    a=np.array(rows)
    np.savetxt(OUTPUT/'recomputed_motion_candidates.csv',a,delimiter=',',header='x_pixel,y_pixel,start_index,end_index,vx_pixel_per_s,vy_pixel_per_s,ncc,peak_rank',comments='')
    return a

def mode_velocity(rows, slow=False):
    speed=np.hypot(rows[:,4],rows[:,5])
    choose=(speed<.18) if slow else (speed>.22)
    a=rows[choose]
    bins=np.linspace(-1,1,101)
    weights=np.maximum(a[:,6]-.35,0)**2/(1+a[:,7])
    hist,xe,ye=np.histogram2d(a[:,4],a[:,5],bins=(bins,bins),weights=weights)
    hist=gaussian_filter(hist,1)
    ix,iy=np.unravel_index(np.argmax(hist),hist.shape)
    center=np.array([(xe[ix]+xe[ix+1])/2,(ye[iy]+ye[iy+1])/2])
    close=np.linalg.norm(a[:,4:6]-center,axis=1)<.09
    v=np.average(a[close,4:6],axis=0,weights=weights[close])
    return v, int(close.sum()),float(np.median(a[close,6]))

def warp(a,v,dt):
    """Translate a reference image by (vx,vy)*dt; no wrap or extrapolation."""
    coords=[yy-v[1]*dt,xx-v[0]*dt]
    w=map_coordinates(np.isfinite(a).astype(float),coords,order=1,mode='constant',cval=0)
    b=map_coordinates(np.nan_to_num(a),coords,order=1,mode='constant',cval=0)
    return np.where(w>.999,b/np.maximum(w,1e-12),np.nan)

def stack_layer(residual,v):
    a=np.array([warp(r,v,-t) for r,t in zip(residual,times)])
    count=np.isfinite(a).sum(axis=0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        template=np.nanmedian(a,axis=0)
    template[count<10]=np.nan
    return template,count



In [ ]:
if REESTIMATE_VELOCITIES:
    candidates = motions()
    va, slow_count, slow_ncc = mode_velocity(candidates, slow=True)
    vc, fast_count, fast_ncc = mode_velocity(candidates, slow=False)
    print('Re-estimated velocities:', va, vc)
else:
    print('Using velocities from the v02 exact-epoch fit.')


## Alternate fast and slow template updates
Each iteration uses all 49 observations. First subtract slow and estimate fast from its aligned median; then subtract fast and estimate slow. A template update requires 10 usable frames. **Where support is insufficient, retain the previous template value** to prevent the boundary erosion found in the original implementation. Such retained values are not new measurements.

The fast template is assigned median zero. Absolute brightness division between the layers remains ambiguous. This boundary correction means results need not exactly reproduce the original uncorrected eight-iteration real-data fit.

Checkpoints at powers of two and the final iteration contain templates and support counts. RMSE uses a fixed region observed in all frames and supported by the first fitted model. The notebook fails rather than silently changing that evaluation region if support is later lost.


In [ ]:
assert ITERATIONS >= 1
def predict(template, velocity):
    return np.array([warp(template, velocity, t) for t in times])

slow, slow_support = stack_layer(I, va)
fast = np.zeros_like(slow)
slowseq = predict(slow, va)
history = []
fixed_mask = None
save_at = {ITERATIONS} | {2**i for i in range(ITERATIONS.bit_length()) if 2**i <= ITERATIONS}
for iteration in range(1, ITERATIONS+1):
    candidate, fast_support = stack_layer(I-slowseq, vc)
    fast = np.where(np.isfinite(candidate), candidate, fast)
    offset = np.nanmedian(fast)
    fast -= offset
    slow += offset
    fastseq = predict(fast, vc)
    candidate, slow_support = stack_layer(I-fastseq, va)
    slow = np.where(np.isfinite(candidate), candidate, slow)
    slowseq = predict(slow, va)
    residual = I-slowseq-fastseq
    if fixed_mask is None:
        fixed_mask = np.all(np.isfinite(residual), axis=0) & data['common_coverage']
        if fixed_mask.sum() == 0: raise RuntimeError('No shared evaluation region')
    if not np.isfinite(residual[:, fixed_mask]).all():
        raise RuntimeError('Fixed evaluation region lost support')
    rmse = float(np.sqrt(np.mean(residual[:, fixed_mask]**2)))
    history.append({'iteration': iteration, 'reconstruction_rmse_kR': rmse,
                    'evaluation_pixels_per_frame': int(fixed_mask.sum())})
    if iteration in save_at:
        np.savez_compressed(OUTPUT / f'checkpoint_{iteration:04d}.npz',
            slow=slow, fast=fast, slow_support=slow_support, fast_support=fast_support,
            slow_velocity=va, fast_velocity=vc, latitude=lat, longitude=lon,
            frame_index=frames, relative_seconds=times, evaluation_mask=fixed_mask)
    if iteration % 8 == 0 or iteration in save_at:
        print(f'Iteration {iteration}: reconstruction RMSE = {rmse:.5f} kRayleigh', flush=True)
        (OUTPUT / 'metrics.json').write_text(json.dumps(history, indent=2))
print('Results saved in', OUTPUT.resolve())


In [ ]:
fig, ax = plt.subplots(figsize=(8,4), layout='constrained')
ax.plot([m['iteration'] for m in history], [m['reconstruction_rmse_kR'] for m in history])
ax.set(xlabel='Completed iterations', ylabel='Reconstruction RMSE (kRayleigh)',
       title='Fixed common coverage; training fit, not independent validation')
ax.grid(alpha=.25)
fig.savefig(OUTPUT / 'error_history.png', dpi=150)
plt.show()


In [ ]:
ITERATION_TO_VIEW = ITERATIONS  # Any saved checkpoint: 1,2,4,8,..., or final.
FRAME_TO_VIEW = 1179            # Zero-based v02 source frame index; see epoch_mapping.csv.
checkpoint = np.load(OUTPUT / f'checkpoint_{ITERATION_TO_VIEW:04d}.npz')
j = int(np.flatnonzero(frames == FRAME_TO_VIEW)[0])
a = warp(checkpoint['slow'], checkpoint['slow_velocity'], times[j])
c = warp(checkpoint['fast'], checkpoint['fast_velocity'], times[j])
r = I[j]-a-c
# Fixed scales allow comparison across checkpoints.
fig, axes = plt.subplots(2,2, figsize=(12,10), layout='constrained')
for ax, arr, title, cmap, low, high in zip(axes.flat, [I[j],a,c,r],
    ['Observed', 'Slow component', 'Fast signed contrast', 'Unexplained residual'],
    ['gray','gray','RdBu_r','RdBu_r'], [lo,lo,-2,-1], [hi,hi,2,1]):
    im=ax.imshow(arr, origin='lower', extent=extent, aspect=aspect, cmap=cmap,vmin=low,vmax=high)
    ax.set(title=title, xlabel='Longitude (degrees E)', ylabel='Latitude (degrees N)')
    fig.colorbar(im,ax=ax,shrink=.75,label='kRayleigh')
fig.suptitle(f'Frame {FRAME_TO_VIEW} | iteration {ITERATION_TO_VIEW}')
fig.savefig(OUTPUT / f'view_{ITERATION_TO_VIEW:04d}_{FRAME_TO_VIEW}.png',dpi=130)
plt.show()


## Optional: inspect the original, unprojected subset
Subset time indices map to v02 and v01 indices in epoch_mapping.csv. The reference time is the exact Epoch of v01 frame 955. The subset preserves original global date metadata, which refers to the full source orbit. Use `source_v02_frame_index` and `Epoch` for the selected interval. To change the projection, start with these original arrays. The runnable separation above uses the supplied fixed-grid frames.


In [ ]:
import netCDF4
with netCDF4.Dataset(DATA_FOLDER / 'original_v02_matched_frames.nc') as original:
    print(original.variables.keys())
    print('Original radiance shape:', original['Radiance'].shape)
    raw_frame_955 = original['Radiance'][25]
    raw_lat_955 = original['Latitude'][25]
    raw_lon_955 = original['Longitude'][25]
